# CKIN2 — analyse globale du portefeuille

Valeur totale et comparaison des caisses, skins, souvenirs, stickers, charms et agents.
Ce notebook lit les CSV sans les modifier. Les valeurs sont des estimations brutes en EUR.

## 1. Chargement et couverture

Lancer la conversion puis `python src/update_market_prices.py home` et `python src/global_snapshots.py home` pour enrichir les données. Les fichiers absents restent signalés ; aucun historique d'inventaire n'est inventé.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# Override only the data directory for reproducible synthetic validation.
DATA_ROOT = Path(os.environ.get("CKIN2_DATA_ROOT", str(PROJECT_ROOT)))
from global_analysis import analyze
result = analyze(DATA_ROOT)
display(result["current"])
display(result["current_detail"][["category", "item_name", "quantity", "price_date", "price_age_days", "market_value"]])


## 2. Valeur actuelle et répartition

Le total complet reste indisponible si une quantité, un prix ou une catégorie manque. Le sous-total connu n'est pas assimilé au total. Un portefeuille vide et intégralement importé vaut zéro.

In [ ]:
current = result["current"]
categories = current.loc[current.category.ne("global")]
if categories.known_value.notna().any():
    px.bar(categories, x="category", y="known_value", hover_data=["total_value", "priced_items", "held_items", "unknown_quantities"], title="Valeur connue par catégorie (EUR ; peut être partielle)").show()
else:
    print("Valorisation indisponible : importer les quantités et les prix de marché.")


## 3. Évolution du portefeuille global

Chaque relevé utilise ses propres quantités. Les changements reflètent à la fois les prix et les entrées/sorties ; ils ne sont pas des rendements d'investissement. Les trous de couverture restent des interruptions de courbe.

In [ ]:
historical = result["historical"]
display(historical)
if not historical.empty:
    px.line(historical, x="snapshot_date", y="total_value", color="category", markers=True, title="Valeurs complètes des snapshots (EUR)").show()
else:
    print("Aucun snapshot global enregistré. Les anciens snapshots de caisses ne décrivent pas tout le portefeuille.")


## 4. Comparaison des prix à panier constant

Indices base 100 à la première date réellement observée commune à tous les objets suivis. Cohorte fixe, poids égaux par objet ; l'indice global pondère également tous les objets et n'est pas la moyenne simple des catégories. Les dates estimées et les acquisitions sont exclues. Un prix antérieur peut être utilisé pendant 45 jours maximum, avec âge et couverture visibles. Sans base commune ou avec une base nulle, la comparaison reste indisponible.

In [ ]:
display(pd.DataFrame([result["diagnostic"]]))
indices = result["indices"]
display(indices)
if not indices.empty:
    px.line(indices, x="date", y="index_100", color="category", markers=True, hover_data=["covered_items", "cohort_items", "oldest_price_age_days"], title="Prix comparés à panier constant — base 100 commune").show()
else:
    print(result["diagnostic"]["reason"])


## 5. Pourquoi la valeur change-t-elle ?

Pour les lignes couvertes aux deux dates : effet prix = quantité précédente × variation de prix ; effet quantité = variation de quantité × prix courant. La somme réconcilie la variation couverte. Les lignes exclues sont affichées et ne valent pas zéro.

In [ ]:
contributions = result["contributions"]
display(contributions)
if not contributions.empty:
    coverage = contributions.groupby(["start_date", "end_date", "category"]).covered.agg(["sum", "count"])
    display(coverage.rename(columns={"sum": "covered_items", "count": "tracked_items"}))
    effects = contributions.groupby(["end_date", "category"])[["price_effect", "quantity_effect", "value_change"]].sum(min_count=1).reset_index()
    display(effects)
    px.bar(effects, x="end_date", y=["price_effect", "quantity_effect"], facet_col="category", title="Contributions couvertes : prix et quantités (EUR)").show()
else:
    print("Deux snapshots globaux sont nécessaires pour expliquer une variation.")


## 6. Limites et prédiction

La cohorte correspond aux objets actuellement suivis, pas à tout le marché CS2. Une évolution du panier impose de recalculer et de documenter la cohorte. Les observations irrégulières ne sont pas des rendements mensuels. Voir `prediction_book.ipynb` pour les expériences de prévision, avec validation sur le passé et comparaison aux prévisions simples. Aucune recommandation d'achat ou de vente n'est produite.